# Advanced Python Exception Handling — Tutorial Problem Set II

## A fresh collection of worked, progressively built problems

**Audience:** intermediate/advanced Python developers. **Runtime:** Python 3.11+; all examples use the standard library. **How to study:** run the notebook from the top, read each short explanation, predict the next code cell, then check the assertions. Each problem introduces a different design situation instead of recycling the previous problem set.

This notebook follows the *instructional rhythm* of the supplied reference lesson—small examples, one new idea at a time, explicit comparisons, and narrative explanations between code cells. The reference primarily introduces handler matching, `try`/`except`/`else`/`finally`, nested handling and EAFP. The new scenarios below extend those foundations, and correct unsafe shortcuts such as broad exception swallowing or implying that `finally` guarantees execution after process termination.

All intentionally failing examples catch the exception *outside* the behavior under examination: **Run All should complete without an uncaught traceback**. Tests are built in and require no third-party packages.

### A compact execution map

- `try` executes the risky operation; matching `except` blocks deal with the *exception from that try body*.
- `else` executes only when the `try` body completed without an exception; exceptions inside `else` are not handled by sibling `except` clauses.
- `finally` executes while leaving the `try` statement normally or via return/exception, subject to unusual exits such as process termination. A return or new exception inside it can replace the original outcome.
- Catch only errors you can handle; let programmer bugs and cancellation propagate. Use `with` when it communicates resource lifetime more clearly.

**Notation:** every problem is independent in intent but uses unique identifiers, so it is also safe to run this notebook sequentially. No files, network access, package installations or real delays are required.

In [1]:
import asyncio
import contextlib
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation, localcontext
import errno
import sys
import traceback
import warnings

print('Python:', sys.version.split()[0])
assert sys.version_info >= (3, 11), 'This notebook needs Python 3.11+ for ExceptionGroup and TaskGroup.'

Python: 3.13.7


---
    ## Problem 01 — An exception raised in `else` escapes its sibling handlers

    **The situation.** An import step parses a record, and a separate callback saves it. A broad `try` could mislabel a callback bug as malformed input.

    **Your task.** Give parsing errors a defined fallback, but allow a callback failure to propagate; prove the execution order.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Observe the branch boundaries

    Build the smallest possible execution trace. Notice that `else` means the `try` body succeeded, *not* that every later operation succeeded.

In [2]:
events01=[]
try:
    number01=int('17')
except ValueError:
    events01.append('parse failed')
else:
    events01.append('parse succeeded')
finally:
    events01.append('finalized')
print(events01)
assert events01 == ['parse succeeded','finalized']

['parse succeeded', 'finalized']


### Step 2 — Define a boundary with one risky expression

    Keep the exception handler around `int(text)` only. A `RuntimeError` from the consumer should not be mistaken for a `ValueError` from parsing.

In [3]:
def parse_and_consume01(text, consume):
    history=[]
    try:
        value=int(text)
    except ValueError:
        history.append('invalid integer')
        return None,history
    else:
        history.append('parsed')
        consume(value)  # Outside the parsing error boundary.
        history.append('consumed')
        return value,history

### Step 3 — Demonstrate what `else` does not catch

    Even if the callback raises a `ValueError`, that error originates in `else` and propagates to the caller. An *outer* handler can decide what to do with it.

In [4]:
def bad_consumer01(value):
    raise ValueError(f'consumer rejected {value}')

try:
    parse_and_consume01('17',bad_consumer01)
except ValueError as exc:
    observed01=str(exc)
print(observed01)
assert observed01 == 'consumer rejected 17'

consumer rejected 17


### Verify the solution

    Check successful parsing, bad input, and callback failure. The failure must not be converted to a `None` result.

In [5]:
saved01=[]
assert parse_and_consume01('9',saved01.append) == (9,['parsed','consumed'])
assert saved01 == [9]
assert parse_and_consume01('not-an-int',saved01.append) == (None,['invalid integer'])
assert saved01 == [9]
try:
    parse_and_consume01('1', bad_consumer01)
except ValueError as exc:
    assert 'consumer rejected' in str(exc)
else:
    raise AssertionError('callback error was swallowed')
print('Problem 01: passed')

Problem 01: passed


### What we learned

    Place the smallest possible operation inside `try`. Use `else` to separate successful parsing from a subsequent action. Exceptions raised in the `else` suite do not enter the sibling `except` suite.

---
    ## Problem 02 — Preserve a traceback with bare `raise`

    **The situation.** An observability wrapper intercepts a failure to record a metric, then immediately propagates it.

    **Your task.** Prove the currently handled exception exists and compare `raise` with `raise exc` without printing uncontrolled tracebacks.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Capture the active exception

    `sys.exception()` returns the exception currently being handled in this thread. It is useful for introspection; do not return it as a substitute for reraising.

In [6]:
def origin02():
    raise LookupError('missing resource')

def active_exception02():
    try:
        origin02()
    except LookupError as exc:
        assert sys.exception() is exc
        return type(exc).__name__,str(exc)

assert active_exception02() == ('LookupError','missing resource')

### Step 2 — Implement the original-traceback-preserving wrapper

    A bare `raise` inside an active handler reraises the current exception. The caller will still see that the wrapper was traversed, but no *extra raise-expression frame* is inserted by `raise exc`.

In [7]:
metrics02=[]
def preserve02():
    try:
        origin02()
    except LookupError:
        metrics02.append('lookup_failed')
        raise

try:
    preserve02()
except LookupError as exc02:
    tb_bare02=[frame.name for frame in traceback.extract_tb(exc02.__traceback__)]
assert metrics02 == ['lookup_failed']
print('bare raise traceback:',tb_bare02)

bare raise traceback: ['<module>', 'preserve02', 'origin02']


### Step 3 — Compare with explicitly raising the name

    `raise exc` explicitly raises that object again and usually adds a traceback location. Count frames in equivalent wrappers instead of relying on source line numbers.

In [8]:
def extra_frame02():
    try:
        origin02()
    except LookupError as exc:
        raise exc

try:
    extra_frame02()
except LookupError as exc02:
    tb_named02=[frame.name for frame in traceback.extract_tb(exc02.__traceback__)]
print('raise exc traceback:',tb_named02)
assert len(tb_named02) >= len(tb_bare02) + 1

raise exc traceback: ['<module>', 'extra_frame02', 'extra_frame02', 'origin02']


### Verify the solution

    Both wrappers preserve the exception type and message; the bare form does not add the explicit new-raise location.

In [9]:
assert tb_bare02[-1] == tb_named02[-1] == 'origin02'
assert tb_bare02.count('preserve02') == 1
assert tb_named02.count('extra_frame02') >= 2
print('Problem 02: passed')

Problem 02: passed


### What we learned

    When an error cannot be resolved, record context and use bare `raise` inside its handler. Avoid `raise exc` unless changing the traceback is intentional.

---
    ## Problem 03 — Translate errors with explicit cause and intentional display

    **The situation.** A parser exposes a stable public exception while the low-level exception retains useful debugging evidence.

    **Your task.** Use `raise ... from exc`, inspect `__cause__` / `__context__`, and contrast it with `from None`.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Define a narrow domain error

    Custom exceptions should expose what a caller can act on. Do not put sensitive input directly in the public message.

In [10]:
class FieldFormatError03(Exception):
    def __init__(self,field):
        self.field=field
        super().__init__(f'invalid format for field {field!r}')

def parse_field03(raw,field):
    try:
        return int(raw)
    except ValueError as exc:
        raise FieldFormatError03(field) from exc

### Step 2 — Inspect chaining without leaking a traceback

    `__cause__` is set by `from exc`. In this explicit-chain example the underlying `ValueError` is also available for debugging.

In [11]:
try:
    parse_field03('NaN','quantity')
except FieldFormatError03 as err03:
    assert err03.field == 'quantity'
    assert isinstance(err03.__cause__,ValueError)
    assert err03.__suppress_context__ is True
    caused03=err03
print('Public:',caused03)
print('Cause type:',type(caused03.__cause__).__name__)

Public: invalid format for field 'quantity'
Cause type: ValueError


### Step 3 — Contrast `from None`

    Suppressing the original traceback in standard display is not the same as erasing the original exception object. The `__context__` remains available.

In [12]:
def parse_quiet03(raw):
    try:
        return int(raw)
    except ValueError:
        raise FieldFormatError03('quantity') from None

try:
    parse_quiet03('NaN')
except FieldFormatError03 as err03:
    quiet03=err03
assert quiet03.__cause__ is None
assert isinstance(quiet03.__context__,ValueError)
assert quiet03.__suppress_context__

### Verify the solution

    Render both tracebacks; an explicit cause is visible while `from None` suppresses its display. Diagnostic strings are generated only for the test.

In [13]:
shown03=''.join(traceback.format_exception(caused03))
hidden03=''.join(traceback.format_exception(quiet03))
assert 'direct cause' in shown03
assert 'During handling of the above exception' not in hidden03
assert 'ValueError' not in hidden03
assert 'FieldFormatError03' in hidden03
print('Problem 03: passed')

Problem 03: passed


### What we learned

    Use `from exc` when callers or diagnostics benefit from the original cause. Use `from None` only as an intentional display decision; it does not guarantee confidentiality of the underlying exception object.

---
    ## Problem 04 — Prevent cleanup from replacing the primary failure

    **The situation.** A function fails during its main operation, and cleanup can also fail. A naive `finally` may hide the first error.

    **Your task.** Understand masking, design an explicit exception policy, and ensure ordinary success still surfaces cleanup failure.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Demonstrate the masking behavior

    The new exception from `finally` becomes the visible exception; the original remains in `__context__`. Avoid this pattern when the original failure is primary.

In [14]:
def naive_cleanup04():
    try:
        raise RuntimeError('operation failed')
    finally:
        raise OSError('cleanup failed')

try:
    naive_cleanup04()
except OSError as exc:
    masked04=exc
assert isinstance(masked04.__context__,RuntimeError)
print('Visible:',masked04,'| earlier:',masked04.__context__)

Visible: cleanup failed | earlier: operation failed


### Step 2 — Create a domain-specific policy

    If both fail, keep the *operation error* as the raised exception and attach a note about cleanup. If only cleanup fails, let the cleanup error propagate. This function simulates two callables without resources.

In [15]:
def operate_and_close04(operate,close):
    try:
        value=operate()
    except Exception as primary:
        try:
            close()
        except Exception as cleanup:
            primary.add_note(f'cleanup also failed: {type(cleanup).__name__}')
        raise
    else:
        close()  # Cleanup failure is the only failure here, so propagate it.
        return value

### Step 3 — Inject operation and cleanup failures independently

    Explicit injection makes it easy to test all four outcomes. In a production resource wrapper, prefer `with` / a context manager when appropriate.

In [16]:
def fail_operation04():
    raise RuntimeError('operation failed')

def fail_cleanup04():
    raise OSError('cleanup failed')

def ok_cleanup04():
    return None

assert operate_and_close04(lambda:7,ok_cleanup04)==7
try:
    operate_and_close04(fail_operation04,fail_cleanup04)
except RuntimeError as exc:
    primary04=exc
assert 'cleanup also failed: OSError' in primary04.__notes__[0]

### Verify the solution

    Inspect operation-only and cleanup-only errors. Neither is lost; errors unrelated to these ordinary exceptions (e.g., cancellation) are not caught by `except Exception`.

In [17]:
try:
    operate_and_close04(fail_operation04,ok_cleanup04)
except RuntimeError as exc:
    assert str(exc)=='operation failed'
else:
    raise AssertionError('operation error missing')
try:
    operate_and_close04(lambda:7,fail_cleanup04)
except OSError as exc:
    assert str(exc)=='cleanup failed'
else:
    raise AssertionError('cleanup error missing')
print('Problem 04: passed')

Problem 04: passed


### What we learned

    `finally` is a language construct, not an automatic error-precedence policy. For critical production cleanup, define explicitly what happens when both the operation and cleanup fail. Our example uses notes to retain a concise secondary failure description.

---
    ## Problem 05 — Map an exception hierarchy to stable API responses

    **The situation.** A command processor raises specialized authorization and input errors. A generic handler placed first would erase distinctions.

    **Your task.** Build a hierarchy, match the specific cases first, and preserve unknown errors for upstream handling.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Design the types

    The base class gives a shared interface; subclasses convey actionable differences without inspecting error strings.

In [18]:
class CommandFailure05(Exception):
    pass
class InvalidCommand05(CommandFailure05):
    pass
class PermissionDenied05(CommandFailure05):
    pass
class QuotaExceeded05(CommandFailure05):
    pass

def execute05(command):
    if command=='invalid': raise InvalidCommand05('bad command')
    if command=='denied': raise PermissionDenied05('no permission')
    if command=='quota': raise QuotaExceeded05('too many requests')
    if command=='bug': raise IndexError('programmer error')
    return 'ok'

### Step 2 — Catch the specific types before the base

    Different callers might use different response status conventions. Here we use symbolic statuses rather than pretending these represent HTTP responses.

In [19]:
def response05(command):
    try:
        result=execute05(command)
    except PermissionDenied05:
        return 'DENIED'
    except InvalidCommand05:
        return 'INVALID'
    except QuotaExceeded05:
        return 'LIMITED'
    except CommandFailure05:
        return 'OTHER_COMMAND_FAILURE'
    else:
        return result

print([response05(x) for x in ('invalid','denied','quota','valid')])

['INVALID', 'DENIED', 'LIMITED', 'ok']


### Step 3 — Reason about catch-all boundaries

    An `IndexError` means the function has a bug. It is not a `CommandFailure05` and should propagate so it is not turned into a bogus user error.

In [20]:
try:
    response05('bug')
except IndexError as exc:
    assert str(exc)=='programmer error'
else:
    raise AssertionError('programmer bug was swallowed')

### Verify the solution

    Prove every application-level branch and the base relationship.

In [21]:
assert [response05(x) for x in ('invalid','denied','quota','valid')] == ['INVALID','DENIED','LIMITED','ok']
assert issubclass(InvalidCommand05,CommandFailure05)
assert issubclass(PermissionDenied05,CommandFailure05)
assert not issubclass(IndexError,CommandFailure05)
print('Problem 05: passed')

Problem 05: passed


### What we learned

    Exception hierarchies establish public contracts. Specific handlers belong before broad handlers, and unknown exceptions should not be silently recast as expected user mistakes.

---
    ## Problem 06 — Implement a context manager with selective suppression

    **The situation.** A best-effort cache lookup may tolerate a *cache-miss exception* but must not hide an actual I/O failure.

    **Your task.** Write `__enter__`/`__exit__` directly, inspect the three exception arguments, and return a Boolean only when suppression is appropriate.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Introduce a resource that records its lifecycle

    The context manager owns no real I/O. Its event log makes the normal and exceptional paths visible.

In [22]:
class MissingCacheEntry06(KeyError):
    pass

class CacheScope06:
    def __init__(self,events):
        self.events=events
    def __enter__(self):
        self.events.append('enter')
        return self
    def __exit__(self,exc_type,exc,tb):
        self.events.append(('exit',exc_type.__name__ if exc_type else None))
        return exc_type is not None and issubclass(exc_type,MissingCacheEntry06)

### Step 2 — Test successful execution and the selected miss

    Returning `False` or `None` from `__exit__` propagates a pending exception. Returning true suppresses it. The class above suppresses only `MissingCacheEntry06` and subclasses.

In [23]:
events06=[]
with CacheScope06(events06):
    events06.append('work')
assert events06==['enter','work',('exit',None)]

events06=[]
with CacheScope06(events06):
    raise MissingCacheEntry06('absent')
assert events06==['enter',('exit','MissingCacheEntry06')]
print('miss suppressed, events:',events06)

miss suppressed, events: ['enter', ('exit', 'MissingCacheEntry06')]


### Step 3 — Test an unexpected error

    An `OSError` should propagate after `__exit__` is called. A context manager that always returned `True` would dangerously hide this failure.

In [24]:
events06=[]
try:
    with CacheScope06(events06):
        raise OSError('disk unavailable')
except OSError as exc:
    assert str(exc)=='disk unavailable'
else:
    raise AssertionError('unexpected error disappeared')
assert events06==['enter',('exit','OSError')]

### Verify the solution

    Also demonstrate normal nested use, showing exit calls in reverse order.

In [25]:
events06=[]
with CacheScope06(events06):
    with CacheScope06(events06):
        events06.append('inside')
assert events06==['enter','enter','inside',('exit',None),('exit',None)]
print('Problem 06: passed')

Problem 06: passed


### What we learned

    An `__exit__` method that returns true changes program control flow by suppressing exceptions. Be restrictive: suppress only a documented, recoverable exception type.

---
    ## Problem 07 — Understand how `@contextmanager` injects exceptions at `yield`

    **The situation.** You want to temporarily modify an application setting and restore it even when work inside the `with` block fails.

    **Your task.** Use a generator-based context manager, follow the exception through `yield`, and avoid accidental suppression.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Build the smallest reversible state change

    `contextlib.contextmanager` converts a generator into a context manager. Everything before `yield` is entry; code after the `yield` resumes on normal exit. The `finally` suite runs when the caller raises too.

In [26]:
settings07={'mode':'safe'}
events07=[]

@contextlib.contextmanager
def temporary_mode07(new_mode):
    old=settings07['mode']
    settings07['mode']=new_mode
    events07.append(('enter',new_mode))
    try:
        yield settings07
    finally:
        settings07['mode']=old
        events07.append(('restore',old))

### Step 2 — Exercise the ordinary exit

    The `yield` value becomes the target of `as`. Restoration must happen after the block, not before.

In [27]:
with temporary_mode07('fast') as visible07:
    assert visible07['mode']=='fast'
    events07.append(('body',visible07['mode']))
assert settings07['mode']=='safe'
assert events07==[('enter','fast'),('body','fast'),('restore','safe')]
print(events07)

[('enter', 'fast'), ('body', 'fast'), ('restore', 'safe')]


### Step 3 — Exercise exceptional exit

    An exception raised within the `with` block is thrown into the generator at `yield`. Because `finally` restores without swallowing, the exception propagates to the surrounding `except`.

In [28]:
events07.clear()
try:
    with temporary_mode07('danger'):
        assert settings07['mode']=='danger'
        raise LookupError('work failed')
except LookupError as exc:
    assert str(exc)=='work failed'
else:
    raise AssertionError('error was swallowed')
assert settings07['mode']=='safe'
assert events07==[('enter','danger'),('restore','safe')]

### Step 4 — See what goes wrong if we accidentally consume the error

    A generator that catches `ValueError` and finishes normally can suppress the block error. Here is the behavior deliberately contained for study; do not copy it into critical cleanup.

In [29]:
@contextlib.contextmanager
def swallowing07():
    try:
        yield
    except ValueError:
        pass  # This causes suppression by finishing the generator normally.

with swallowing07():
    raise ValueError('silenced')
print('The block continued after the deliberately suppressed error.')

The block continued after the deliberately suppressed error.


### Verify the solution

    Run nested setting changes to check the stack-like restore order.

In [30]:
events07.clear()
with temporary_mode07('outer'):
    with temporary_mode07('inner'):
        assert settings07['mode']=='inner'
    assert settings07['mode']=='outer'
assert settings07['mode']=='safe'
assert events07==[('enter','outer'),('enter','inner'),('restore','outer'),('restore','safe')]
print('Problem 07: passed')

Problem 07: passed


### What we learned

    `@contextmanager` changes exception flow: the block exception is thrown into the generator at `yield`. Use `try/finally` to restore state and let errors propagate unless suppression is explicitly intended.

---
    ## Problem 08 — Analyze nested `ExceptionGroup` with `.split()`

    **The situation.** A parallel validation subsystem reports failures from several branches in a nested `ExceptionGroup`.

    **Your task.** Partition the group by exception type, preserve its tree structure, and handle one subset with `except*`.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Make a nontrivial error tree

    The leaves belong to two categories and are nested by processing batch. This is a static example; it avoids timing-dependent failures.

In [31]:
group08=ExceptionGroup('all batches',[
    ExceptionGroup('batch A',[ValueError('A:bad value'),TypeError('A:wrong type')]),
    ExceptionGroup('batch B',[ValueError('B:bad value'),LookupError('B:not found')]),
])
print('Total immediate branches:',len(group08.exceptions))
assert len(group08.exceptions)==2

Total immediate branches: 2


### Step 2 — Partition without flattening

    `.split(ValueError)` returns `(matching, remaining)`. Each is an `ExceptionGroup` or `None`; subgroup nesting is preserved for included leaves.

In [32]:
value_errors08,other_errors08=group08.split(ValueError)
assert value_errors08 is not None and other_errors08 is not None
assert len(value_errors08.exceptions)==2
assert all(isinstance(branch,ExceptionGroup) for branch in value_errors08.exceptions)
assert all(len(branch.exceptions)==1 for branch in value_errors08.exceptions)
print('Value-error subtree:',value_errors08)

Value-error subtree: all batches (2 sub-exceptions)


### Step 3 — Implement a controlled flattening helper for reporting

    Flattening is fine for a *summary*. Keep the original group available for structured handling, because subgroup boundaries carry useful context.

In [33]:
def leaves08(exc):
    if isinstance(exc,BaseExceptionGroup):
        for child in exc.exceptions:
            yield from leaves08(child)
    else:
        yield exc

assert sorted(str(e) for e in leaves08(value_errors08))==['A:bad value','B:bad value']
assert {type(e).__name__ for e in leaves08(other_errors08)}=={'TypeError','LookupError'}

### Step 4 — Use `except*` without misrepresenting partial handling

    The first handler receives only matching leaves. The second receives *everything else*. You cannot combine `except` and `except*` in a single try statement.

In [34]:
handled08=[]
try:
    raise group08
except* ValueError as group_of_values:
    handled08.extend(('bad-value',str(e)) for e in leaves08(group_of_values))
except* Exception as rest:
    handled08.extend(('other',type(e).__name__) for e in leaves08(rest))
print(handled08)

[('bad-value', 'A:bad value'), ('bad-value', 'B:bad value'), ('other', 'TypeError'), ('other', 'LookupError')]


### Verify the solution

    Check exact leaf coverage: no dropped, repeated, or fabricated failures.

In [35]:
assert len(handled08)==4
assert sum(kind=='bad-value' for kind,_ in handled08)==2
assert {name for kind,name in handled08 if kind=='other'}=={'TypeError','LookupError'}
assert len(list(leaves08(group08)))==4
print('Problem 08: passed')

Problem 08: passed


### What we learned

    `ExceptionGroup` lets independent errors travel together. `.split()` and `except*` preserve meaningful nested structure while targeting selected leaf types; use explicit flattening only for presentation.

---
    ## Problem 09 — Render diagnostic tracebacks without exposing secrets

    **The situation.** A lower-level parser includes a secret-bearing raw record in its error text. User-facing diagnostics must reveal only a stable public message.

    **Your task.** Create an explicit error boundary, compare chained and unchained formatting, and avoid capturing local variables.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Simulate a sensitive lower-level failure

    This controlled example models a hostile or overly verbose third-party error that includes secret data in `str(exc)`.

In [36]:
SECRET09='api-key-DO-NOT-LOG'
class PublicRecordError09(Exception):
    pass

def parse_sensitive09(raw):
    try:
        raise ValueError(f'bad record: {raw}')
    except ValueError as exc:
        raise PublicRecordError09('record cannot be processed') from exc

### Step 2 — Catch the exception at a trust boundary

    The exception carries useful debugging information in its cause, so presenting its full traceback is a policy choice. First, inspect the danger in a controlled test variable.

In [37]:
try:
    parse_sensitive09(SECRET09)
except PublicRecordError09 as exc:
    public_exc09=exc

full09=''.join(traceback.format_exception(public_exc09))
assert SECRET09 in full09
assert str(public_exc09)=='record cannot be processed'
print('Full chained traceback contains secret:',SECRET09 in full09)

Full chained traceback contains secret: True


### Step 3 — Format only the safe public exception

    `chain=False` excludes the cause from display; `capture_locals=False` avoids automatically dumping local variables. This controls *this output*, not all possible logging or exception disclosure.

In [38]:
safe09=''.join(traceback.TracebackException.from_exception(
    public_exc09,capture_locals=False).format(chain=False))
assert 'PublicRecordError09' in safe09
assert SECRET09 not in safe09
print('Safe traceback suffix:',safe09.splitlines()[-1])

Safe traceback suffix: PublicRecordError09: record cannot be processed


### Step 4 — Choose a production-facing summary

    A stable category and generic message are safer than returning an exception object or its full formatted traceback to clients.

In [39]:
def public_summary09(exc):
    if isinstance(exc,PublicRecordError09):
        return {'code':'INVALID_RECORD','message':'record cannot be processed'}
    raise TypeError('unexpected exception type')

summary09=public_summary09(public_exc09)
assert SECRET09 not in repr(summary09)
print(summary09)

{'code': 'INVALID_RECORD', 'message': 'record cannot be processed'}


### Verify the solution

    Ensure the public representation stays secret-free while internal debugging retains the original cause type.

In [40]:
assert isinstance(public_exc09.__cause__,ValueError)
assert SECRET09 in str(public_exc09.__cause__)
assert SECRET09 not in str(public_exc09)
assert SECRET09 not in safe09
assert SECRET09 not in repr(summary09)
print('Problem 09: passed')

Problem 09: passed


### What we learned

    Exception chaining and exception rendering serve different audiences. Keep internal causes when useful, but explicitly control traceback formatting and public error messages. `chain=False` alone is not a universal secret-redaction system.

---
    ## Problem 10 — Handle filesystem failures by `errno`, not message text

    **The situation.** A file loader can return a default if the file is missing; permission and device failures must remain errors.

    **Your task.** Use the `OSError` hierarchy and its `errno` attribute, with a deterministic injectable read function.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — See what the error data actually contains

    System error messages vary by platform and locale. `errno.ENOENT` is a symbolic constant for missing path; do not parse the human-readable message.

In [41]:
missing10=FileNotFoundError(errno.ENOENT,'not found','/fake/path')
assert isinstance(missing10,OSError)
assert missing10.errno==errno.ENOENT
print(type(missing10).__name__,missing10.errno)

FileNotFoundError 2


### Step 2 — Implement a narrow fallback

    The function catches only `OSError` from the `reader` call. Other failures, including programmer errors, propagate. `errno` is checked explicitly instead of swallowing every I/O error.

In [42]:
def read_optional10(reader,default=''):
    try:
        payload=reader()
    except OSError as exc:
        if exc.errno==errno.ENOENT:
            return default
        raise
    else:
        return payload

### Step 3 — Inject a missing file and a denied file

    We simulate errors directly, making the test portable and avoiding real filesystem privileges.

In [43]:
def missing_reader10():
    raise FileNotFoundError(errno.ENOENT,'missing','/fake')

def denied_reader10():
    raise PermissionError(errno.EACCES,'permission denied','/fake')

assert read_optional10(lambda:'payload')=='payload'
assert read_optional10(missing_reader10,default='fallback')=='fallback'
try:
    read_optional10(denied_reader10)
except PermissionError as exc:
    assert exc.errno==errno.EACCES
else:
    raise AssertionError('permission failure was swallowed')

### Verify the solution

    Check that an unexpected programming error is not silently mistaken for an unavailable file.

In [44]:
def bug_reader10():
    raise TypeError('bad reader implementation')
try:
    read_optional10(bug_reader10)
except TypeError as exc:
    assert str(exc)=='bad reader implementation'
else:
    raise AssertionError('programmer bug was swallowed')
print('Problem 10: passed')

Problem 10: passed


### What we learned

    Recover from a specific, documented operating-system condition (`ENOENT`). A blanket `except OSError` that returns a default for all error numbers could hide permission problems, damaged storage or other important failures.

---
    ## Problem 11 — Handle generator termination without triggering PEP 479

    **The situation.** A streaming parser wants to stop after a sentinel. A developer incorrectly raises `StopIteration` from inside the generator.

    **Your task.** Observe the resulting `RuntimeError`, use `return` for normal termination, and distinguish a normal iterator from a generator function.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Demonstrate the incorrect generator

    PEP 479 converts an accidental `StopIteration` escaping generator code into `RuntimeError`, keeping a generator bug from looking like normal iteration completion.

In [45]:
def broken_stream11():
    yield 'first'
    raise StopIteration('not the right way to stop a generator')

it11=broken_stream11()
assert next(it11)=='first'
try:
    next(it11)
except RuntimeError as exc:
    broken_error11=exc
assert isinstance(broken_error11.__cause__,StopIteration)
print(type(broken_error11).__name__,str(broken_error11))

RuntimeError generator raised StopIteration


### Step 2 — Replace the exception with `return`

    A generator `return` signals exhaustion. Its optional value becomes the `StopIteration.value` carried internally when iterated manually.

In [46]:
def correct_stream11():
    yield 'first'
    return 'stream complete'

it11=correct_stream11()
assert next(it11)=='first'
try:
    next(it11)
except StopIteration as exc:
    completion11=exc.value
assert completion11=='stream complete'
print('Generator return value:',completion11)

Generator return value: stream complete


### Step 3 — Contrast with a custom iterator

    An iterator class `__next__` method *should* raise `StopIteration` to report exhaustion. The issue concerns the exception escaping the body of a generator function.

In [47]:
class TwoItems11:
    def __init__(self):
        self.index=0
    def __iter__(self):
        return self
    def __next__(self):
        if self.index>=2:
            raise StopIteration
        self.index+=1
        return self.index

assert list(TwoItems11())==[1,2]

### Verify the solution

    The corrected generator works naturally with `list()` and `for`, without exposing its optional return value as an element.

In [48]:
assert list(correct_stream11())==['first']
assert [x for x in TwoItems11()]==[1,2]
try:
    list(broken_stream11())
except RuntimeError as exc:
    assert isinstance(exc.__cause__,StopIteration)
else:
    raise AssertionError('broken generator did not show its bug')
print('Problem 11: passed')

Problem 11: passed


### What we learned

    Raise `StopIteration` from an iterator’s `__next__`, but use `return` to finish a generator. Do not catch and silently convert unrelated generator failures into end-of-stream.

---
    ## Problem 12 — Apply EAFP without accidentally caching failed lookups

    **The situation.** A cache stores successful results, including `None`. A slow factory may raise `KeyError` as a genuine bug while computing a new value.

    **Your task.** Use one narrow lookup handler and keep factory invocation *outside* the lookup exception handler.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Write the tempting but incorrect version

    This incorrect implementation treats a `KeyError` from the factory as if it were a cache miss, which can mask the real defect. Keep it as a discussion example; do not use it.

In [49]:
def incorrect_cache12(cache,key,factory):
    try:
        return cache[key]
    except KeyError:
        return factory(key)  # This is outside the handler body but still not cached.

# The issue in this *simplified* version is lack of caching, not swallowing a
# factory KeyError. A true swallowing bug requires factory() inside the try.
def dangerous_cache12(cache,key,factory):
    try:
        return cache[key] if key in cache else factory(key)
    except KeyError:
        return 'incorrect fallback'

### Step 2 — Define the correct contract clearly

    A hit returns even when the cached value is `None`. A miss computes once; only a successful computation is stored. If factory fails, the cache remains unchanged.

In [50]:
def memoize12(cache,key,factory):
    try:
        return cache[key]
    except KeyError:
        pass
    value=factory(key)
    cache[key]=value
    return value

### Step 3 — Prove `None` is a legitimate cached value

    Never use `dict.get()` without a sentinel if `None` can be an actual value and you need to distinguish it from absence.

In [51]:
cache12={'known':None}
calls12=[]
def factory12(key):
    calls12.append(key)
    return len(key)
assert memoize12(cache12,'known',factory12) is None
assert calls12==[]
assert memoize12(cache12,'new',factory12)==3
assert memoize12(cache12,'new',factory12)==3
assert calls12==['new']
print(cache12)

{'known': None, 'new': 3}


### Step 4 — Force a real factory error

    The factory accesses an absent key in a different dictionary and raises `KeyError`. That error is not a cache miss; it should propagate.

In [52]:
def faulty_factory12(key):
    return {'different':'value'}[key]

before12=dict(cache12)
try:
    memoize12(cache12,'bad',faulty_factory12)
except KeyError as exc:
    assert exc.args==('bad',)
else:
    raise AssertionError('factory bug was swallowed')
assert cache12==before12

### Verify the solution

    Compare the incorrect approach: its fallback hides precisely the bug the correct code surfaces.

In [53]:
assert dangerous_cache12({},'bad',faulty_factory12)=='incorrect fallback'
try:
    memoize12({},'bad',faulty_factory12)
except KeyError:
    pass
else:
    raise AssertionError('factory failure did not propagate')
print('Problem 12: passed')

Problem 12: passed


### What we learned

    EAFP is not an invitation to wrap large blocks in one `try`. Isolate the exact cache lookup. A failed factory must not be interpreted as an absent cache entry, and caching should occur only after successful computation.

---
    ## Problem 13 — Collect exceptions from concurrent futures without losing results

    **The situation.** Several independent calculations run in a thread pool. A failure in one input should not prevent collecting other results.

    **Your task.** Use `Future.result()` to retrieve either a value or the original exception; record results by input identity, not completion timing.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Define a deterministic worker

    There is no real network or sleep here. The function either computes a square or raises a domain-specific exception based solely on its argument.

In [54]:
class CalculationError13(Exception):
    pass

def square13(number):
    if number<0:
        raise CalculationError13(f'negative input: {number}')
    return number*number

assert square13(4)==16
try:
    square13(-1)
except CalculationError13 as exc:
    assert 'negative input' in str(exc)

### Step 2 — Submit all work before inspecting results

    `executor.submit` returns `Future` objects. In this design, a worker exception is captured by its future and reraised by `.result()` in the caller.

In [55]:
inputs13=[3,-2,4,-5,0]
with ThreadPoolExecutor(max_workers=3) as pool13:
    futures13={n:pool13.submit(square13,n) for n in inputs13}
    # Calling result() in the input order gives stable reporting order.
    print('Submitted:',list(futures13))
    assert len(futures13)==5

Submitted: [3, -2, 4, -5, 0]


### Step 3 — Collect successes and failures separately

    A local handler catches only `CalculationError13`. Other exceptions are not expected under this worker contract and should propagate.

In [56]:
def run_batch13(inputs):
    successes={}
    failures={}
    with ThreadPoolExecutor(max_workers=3) as pool:
        futures=[(position,pool.submit(square13,value))
                 for position,value in enumerate(inputs)]
        for position,future in futures:
            try:
                successes[position]=future.result()
            except CalculationError13 as exc:
                failures[position]=exc
    return successes,failures

success13,fail13=run_batch13(inputs13)
print('successes:',success13)
print('failures:',{i:str(e) for i,e in fail13.items()})

successes: {0: 9, 2: 16, 4: 0}
failures: {1: 'negative input: -2', 3: 'negative input: -5'}


### Step 4 — Explain the indexing choice

    Two equal input values are different jobs. The solution uses position as identity instead of a dictionary keyed by the raw input, which would lose duplicates.

In [57]:
duplicate13=[2,2,-1,-1]
success_dup13,fail_dup13=run_batch13(duplicate13)
assert success_dup13=={0:4,1:4}
assert set(fail_dup13)=={2,3}

### Verify the solution

    Verify all five results were accounted for and each error preserved its type.

In [58]:
assert success13=={0:9,2:16,4:0}
assert set(fail13)=={1,3}
assert all(isinstance(exc,CalculationError13) for exc in fail13.values())
assert len(success13)+len(fail13)==len(inputs13)
print('Problem 13: passed')

Problem 13: passed


### What we learned

    A `Future` preserves the worker exception until `.result()` is called. Collect per-job outcomes deliberately. A thread-pool context manager waits for submitted work to finish when exiting; avoid timing-dependent tests when teaching exception semantics.

---
    ## Problem 14 — Use `asyncio.TaskGroup` to clean up a failing sibling

    **The situation.** An asynchronous pipeline starts a long-running sibling and a second task that fails. The sibling must be cancelled and run its cleanup.

    **Your task.** Trace cancellation into `finally`, catch the resulting `ExceptionGroup`, and avoid accidentally swallowing `CancelledError`.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Build the sibling and failing task

    The sibling signals when it has started. The failing task waits for that signal, so its failure cannot race ahead of sibling setup.

In [59]:
async def run_group14():
    events=[]
    sibling_ready=asyncio.Event()
    async def sibling():
        try:
            events.append('sibling started')
            sibling_ready.set()
            await asyncio.Event().wait()  # Wait forever unless cancelled.
        finally:
            events.append('sibling cleaned up')
    async def fails():
        await sibling_ready.wait()
        events.append('failure raised')
        raise ValueError('task rejected record')
    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(sibling())
            group.create_task(fails())
    except* ValueError as errors:
        events.append(('caught',tuple(str(e) for e in errors.exceptions)))
    return events

### Step 2 — Execute using notebook-native `await`

    Modern Jupyter supports top-level `await`, so there is no nested `asyncio.run()` call inside its existing event loop.

In [60]:
events14=await run_group14()
print(events14)
assert 'sibling started' in events14
assert 'failure raised' in events14
assert 'sibling cleaned up' in events14

['sibling started', 'failure raised', 'sibling cleaned up', ('caught', ('task rejected record',))]


### Step 3 — Inspect cleanup ordering

    `TaskGroup` cancels remaining tasks and waits for them to finish before it surfaces the group of non-cancellation failures.

In [61]:
assert events14.index('sibling started')<events14.index('failure raised')
assert events14.index('failure raised')<events14.index('sibling cleaned up')
assert events14[-1]==('caught',('task rejected record',))

### Verify the solution

    Run the same deterministic scenario again to confirm there is no leaked task or dependence on elapsed wall-clock time.

In [62]:
again14=await run_group14()
assert again14==events14
print('Problem 14: passed')

Problem 14: passed


### What we learned

    `TaskGroup` provides structured concurrency: sibling tasks are cancelled and awaited if one task fails, and non-cancellation failures are surfaced as an exception group. Cleanup belongs in `finally`; cancellation should normally propagate.

---
    ## Problem 15 — Distinguish an owned timeout from external cancellation

    **The situation.** A service may impose a timeout on an await, but an external caller may also cancel the task.

    **Your task.** Translate only the timeout you own. Keep `asyncio.CancelledError` visible to the caller and prove cleanup happens in both cases.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Build a cancellable service

    The `finally` suite records cleanup when an await stops for either reason. Waiting on an unset event avoids a busy loop.

In [63]:
async def waiting_service15(events,started=None):
    try:
        events.append('work started')
        if started is not None:
            started.set()
        await asyncio.Event().wait()
    finally:
        events.append('work cleaned up')

### Step 2 — Handle an explicitly owned deadline

    `asyncio.timeout()` translates the cancellation it initiates into `TimeoutError` *outside* the context. The timeout duration is short and the underlying wait is event-based.

In [64]:
async def owned_timeout15():
    events=[]
    try:
        async with asyncio.timeout(0.01):
            await waiting_service15(events)
    except TimeoutError:
        events.append('deadline exceeded')
    return events

timeout_events15=await owned_timeout15()
assert timeout_events15==['work started','work cleaned up','deadline exceeded']
print(timeout_events15)

['work started', 'work cleaned up', 'deadline exceeded']


### Step 3 — Simulate a caller cancelling the service

    Start a task, wait for its own readiness signal, then cancel it. `CancelledError` is a `BaseException` subclass in modern Python; do not convert it to a generic timeout.

In [65]:
async def external_cancel15():
    events=[]
    started=asyncio.Event()
    task=asyncio.create_task(waiting_service15(events,started))
    await started.wait()
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        events.append('external cancellation observed')
    else:
        raise AssertionError('cancellation disappeared')
    return events

cancel_events15=await external_cancel15()
print(cancel_events15)

['work started', 'work cleaned up', 'external cancellation observed']


### Verify the solution

    Both paths restore resources, but they carry different public meanings.

In [66]:
assert cancel_events15==['work started','work cleaned up','external cancellation observed']
assert timeout_events15[-1]=='deadline exceeded'
assert 'deadline exceeded' not in cancel_events15
print('Problem 15: passed')

Problem 15: passed


### What we learned

    Own the deadline you create; catch `TimeoutError` outside its timeout context. External cancellation represents a separate control-flow decision and should not be reported as a service timeout.

---
    ## Problem 16 — Create strict decimal validation using local exception traps

    **The situation.** A payment input must be finite, nonnegative, and representable exactly in cents. Accepting arbitrary binary floats would introduce ambiguity.

    **Your task.** Validate types explicitly, use `Decimal`, and convert decimal precision failures into one documented error.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Define a public error type and a fixed precision target

    `Decimal` construction from strings avoids first converting through a binary float. `is_finite()` catches NaN and infinity, which are not sensible payment amounts.

In [67]:
from decimal import Inexact

class MoneyFormatError16(ValueError):
    pass
CENT16=Decimal('0.01')

def decimal_input16(raw):
    if not isinstance(raw,(str,Decimal)):
        raise MoneyFormatError16('use a string or Decimal, not a float')
    try:
        value=Decimal(raw)
    except InvalidOperation as exc:
        raise MoneyFormatError16('invalid decimal text') from exc
    if not value.is_finite() or value<0:
        raise MoneyFormatError16('amount must be finite and nonnegative')
    return value

### Step 2 — Use a local context to detect rounding

    `quantize` would ordinarily round an amount like `1.005`. By trapping `Inexact`, we reject inputs that cannot be represented as exact cents. Local context avoids changing global decimal behavior.

In [68]:
def parse_amount16(raw):
    value=decimal_input16(raw)
    try:
        with localcontext() as ctx:
            ctx.prec=28
            ctx.traps[Inexact]=True
            cents=value.quantize(CENT16)
    except (Inexact,InvalidOperation) as exc:
        raise MoneyFormatError16('more than two significant decimal places or too large') from exc
    return cents

assert parse_amount16('1.20')==Decimal('1.20')
assert parse_amount16(Decimal('0'))==Decimal('0.00')

### Step 3 — Compare valid and invalid inputs

    A trailing zero beyond two decimal places is not an *inexact* value and is allowed: `1.230` equals `1.23`. An actual half-cent is rejected.

In [69]:
assert parse_amount16('1.230')==Decimal('1.23')
for invalid16 in ('1.005','NaN','Infinity','-0.01','hello',1.25,True):
    try:
        parse_amount16(invalid16)
    except MoneyFormatError16:
        pass
    else:
        raise AssertionError(f'accepted invalid input {invalid16!r}')
print('Rejected all seven invalid inputs')

Rejected all seven invalid inputs


### Step 4 — Keep the consumer in exact arithmetic

    After validation, ordinary decimal addition can implement a transparent bill total. Business rounding policy (if needed) must be chosen separately rather than silently introduced here.

In [70]:
def total16(raw_amounts):
    return sum((parse_amount16(v) for v in raw_amounts),Decimal('0.00'))

assert total16(['0.10','0.20','1.23'])==Decimal('1.53')
print('Exact total:',total16(['0.10','0.20','1.23']))

Exact total: 1.53


### Verify the solution

    Distinguish formatting errors from unexpected coding errors and retain the underlying decimal cause when available.

In [71]:
try:
    parse_amount16('0.001')
except MoneyFormatError16 as exc:
    assert isinstance(exc.__cause__,Inexact)
try:
    parse_amount16('not-money')
except MoneyFormatError16 as exc:
    assert isinstance(exc.__cause__,InvalidOperation)
assert parse_amount16('999.99')==Decimal('999.99')
print('Problem 16: passed')

Problem 16: passed


### What we learned

    Exception handling can enforce numerical correctness. Choose an explicit input contract, localize decimal traps, and translate only anticipated decimal failures. A monetary rounding policy must be explicit when rounding is actually desired.

---
    ## Problem 17 — Turn warnings into exceptions only inside a bounded test

    **The situation.** A migration tool emits a deprecation warning. Your test suite treats one warning as a failure without globally altering warning behavior for the notebook.

    **Your task.** Use `warnings.catch_warnings()` to scope the filter and distinguish warnings from ordinary exceptions.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Create a legacy API

    `warnings.warn` emits a warning, which normally is not an exception. The `stacklevel` makes its reported location point toward the call site.

In [72]:
def legacy_api17():
    warnings.warn('old_api will be removed',DeprecationWarning,stacklevel=2)
    return 42

### Step 2 — Record the warning without failing

    Inside `catch_warnings(record=True)`, collected warning messages can be asserted without printing a warning into unrelated notebook output.

In [73]:
with warnings.catch_warnings(record=True) as captured17:
    warnings.simplefilter('always',DeprecationWarning)
    answer17=legacy_api17()
assert answer17==42
assert len(captured17)==1
assert captured17[0].category is DeprecationWarning
print(str(captured17[0].message))

old_api will be removed


### Step 3 — Turn a targeted warning into an error

    The filter matches only `DeprecationWarning` messages starting with `old_api`. The test catches it as an exception; the filter is restored automatically when the context exits.

In [74]:
with warnings.catch_warnings():
    warnings.filterwarnings('error',message=r'^old_api',category=DeprecationWarning)
    try:
        legacy_api17()
    except DeprecationWarning as exc:
        assert str(exc)=='old_api will be removed'
    else:
        raise AssertionError('target warning was not promoted')

### Step 4 — Check that the filter is not leaked

    Use another local recording context: after the preceding context exits, the same call should simply emit a recorded warning.

In [75]:
with warnings.catch_warnings(record=True) as after17:
    warnings.simplefilter('always',DeprecationWarning)
    assert legacy_api17()==42
assert len(after17)==1

### Verify the solution

    Verify a nonmatching warning stays a warning under the targeted filter.

In [76]:
with warnings.catch_warnings(record=True) as mixed17:
    warnings.simplefilter('always')
    warnings.filterwarnings('error',message=r'^old_api',category=DeprecationWarning)
    warnings.warn('other_api is stable',DeprecationWarning)
assert len(mixed17)==1
assert str(mixed17[0].message)=='other_api is stable'
print('Problem 17: passed')

Problem 17: passed


### What we learned

    Warnings convey advisory information; exceptions alter control flow. Promote targeted warnings to errors only inside a scoped testing or migration policy, not by unexpectedly modifying global application warning settings.

---
    ## Problem 18 — Build an atomic mini-language interpreter with multi-error reports

    **The situation.** A stockroom accepts a batch of `ADD SKU QTY` and `REMOVE SKU QTY` commands. A malformed line or negative resulting stock must prevent **all** mutations.

    **Your task.** Separate syntax, business validation, and commit; accumulate line-numbered errors without losing the original stock state.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Design an exception that includes stable line context

    Parse errors should be independently reportable. We store a line number and a simple error code instead of dumping arbitrary raw input into error messages.

In [77]:
class StockCommandError18(ValueError):
    def __init__(self,line,code):
        self.line=line
        self.code=code
        super().__init__(f'line {line}: {code}')

assert str(StockCommandError18(4,'BAD_QUANTITY'))=='line 4: BAD_QUANTITY'

### Step 2 — Parse exactly one command

    First validate command shape, then validate the operation and quantity. Do not catch arbitrary exceptions from the rest of the application under the label "malformed command".

In [78]:
def parse_command18(text,line):
    pieces=text.split()
    if len(pieces)!=3:
        raise StockCommandError18(line,'WRONG_FIELD_COUNT')
    operation,sku,raw_quantity=pieces
    if operation not in {'ADD','REMOVE'} or not sku.isidentifier():
        raise StockCommandError18(line,'BAD_OPERATION_OR_SKU')
    try:
        quantity=int(raw_quantity)
    except ValueError as exc:
        raise StockCommandError18(line,'BAD_QUANTITY') from exc
    if quantity<=0:
        raise StockCommandError18(line,'NONPOSITIVE_QUANTITY')
    return operation,sku,quantity

### Step 3 — Collect syntax problems without abandoning later lines

    A command line is independent of other lines at the syntax layer. Skip blank lines or `#` comment lines and collect all malformed lines together.

In [79]:
def parse_batch18(lines):
    parsed=[]
    errors=[]
    for line_number,text in enumerate(lines,start=1):
        if not text.strip() or text.lstrip().startswith('#'):
            continue
        try:
            instruction=parse_command18(text,line_number)
        except StockCommandError18 as exc:
            errors.append(exc)
        else:
            parsed.append((line_number,instruction))
    return parsed,errors

parsed18,parse_errors18=parse_batch18(['ADD pens 3','REMOVE pens nope','#skip','ADD erasers 1'])
assert len(parsed18)==2
assert [(e.line,e.code) for e in parse_errors18]==[(2,'BAD_QUANTITY')]

### Step 4 — Simulate semantic changes on a copy

    The working dictionary is a copy, so even after multiple valid commands a failure never mutates the original stock. A negative stock balance is a domain failure, not a parser failure.

In [80]:
def validate_changes18(initial,parsed):
    staged=dict(initial)
    issues=[]
    for line,(operation,sku,quantity) in parsed:
        original=staged.get(sku,0)
        proposed=original+quantity if operation=='ADD' else original-quantity
        if proposed<0:
            issues.append(StockCommandError18(line,'INSUFFICIENT_STOCK'))
        else:
            staged[sku]=proposed
    return staged,issues

### Step 5 — Establish the all-or-nothing commit boundary

    Combine syntax and semantic problems into a single `ExceptionGroup`. Only after the group is empty do we replace the caller-owned dictionary contents. This is atomic with respect to validation logic in this single-threaded example, not a cross-thread/database transaction.

In [81]:
def apply_batch18(stock,lines):
    parsed,syntax_errors=parse_batch18(lines)
    staged,semantic_errors=validate_changes18(stock,parsed)
    problems=syntax_errors+semantic_errors
    if problems:
        raise ExceptionGroup('stock batch rejected',problems)
    stock.clear()
    stock.update(staged)
    return stock

### Step 6 — Inspect an invalid batch from the caller side

    Use `except*` so the test inspects every domain exception. Validate *immutability on failure* as well as the number and location of problems.

In [82]:
stock18={'pens':2}
before18=dict(stock18)
issues18=[]
try:
    apply_batch18(stock18,['ADD pens 3','REMOVE pens 100','ADD clips invalid','ADD erasers 5'])
except* StockCommandError18 as group:
    issues18.extend((e.line,e.code) for e in group.exceptions)
assert stock18==before18
assert issues18==[(3,'BAD_QUANTITY'),(2,'INSUFFICIENT_STOCK')]
print('Rejected with:',issues18,'| original remains:',stock18)

Rejected with: [(3, 'BAD_QUANTITY'), (2, 'INSUFFICIENT_STOCK')] | original remains: {'pens': 2}


### Step 7 — Commit a completely valid batch

    Successful later commands see earlier staged changes. The caller receives the same dictionary object it passed in, now updated.

In [83]:
original_id18=id(stock18)
returned18=apply_batch18(stock18,['ADD pens 3','REMOVE pens 4','ADD clips 7'])
assert id(returned18)==original_id18
assert stock18=={'pens':1,'clips':7}
print('Committed:',stock18)

Committed: {'pens': 1, 'clips': 7}


### Verify the solution

    Verify three important boundaries: empty input, negative stock, and a parser error with an explicit cause.

In [84]:
empty18={'a':2}
assert apply_batch18(empty18,['','# comment'])=={'a':2}
try:
    parse_command18('ADD a not_a_number',10)
except StockCommandError18 as exc:
    assert exc.line==10 and exc.code=='BAD_QUANTITY'
    assert isinstance(exc.__cause__,ValueError)
try:
    apply_batch18(empty18,['REMOVE a 3'])
except* StockCommandError18 as group:
    assert group.exceptions[0].code=='INSUFFICIENT_STOCK'
assert empty18=={'a':2}
print('Problem 18: passed')

Problem 18: passed


### What we learned

    Robust batch processing needs an explicit transaction boundary: parse, gather diagnostic errors, simulate the full operation on a separate state, and mutate shared state only if all validation succeeds. The exception group reports multiple actionable failures without partially applying the batch.

---
    ## Problem 19 — Avoid hiding descriptor bugs with `getattr(..., default)`

    **The situation.** A dynamically computed property raises `AttributeError` internally. A caller incorrectly treats the error as proof that the property does not exist.

    **Your task.** Contrast missing attributes with failing descriptors and design an explicit getter API that does not confuse the two.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Produce an error inside a real property

    A property can itself raise `AttributeError` because of a bug or a missing nested dependency. `getattr(obj, name, default)` may return the default in both cases.

In [85]:
class Profile19:
    def __init__(self):
        self.data={'name':'Ada'}
    @property
    def display_name(self):
        return self.data['missing_nested'].upper()  # KeyError, currently.
    @property
    def broken_attribute(self):
        return self.nonexistent_nested.name  # Raises AttributeError internally.

profile19=Profile19()
assert getattr(profile19,'does_not_exist','DEFAULT')=='DEFAULT'
assert getattr(profile19,'broken_attribute','DEFAULT')=='DEFAULT'
print('The default hid a real property error.')

The default hid a real property error.


### Step 2 — Show what direct access reveals

    Without the default, the nested attribute error propagates and its message points to the actual missing dependency.

In [86]:
try:
    profile19.broken_attribute
except AttributeError as exc:
    assert 'nonexistent_nested' in str(exc)
else:
    raise AssertionError('descriptor failure disappeared')

### Step 3 — Define a narrow domain accessor

    For attributes owned by the model, prefer a dedicated data mapping with a clear missing-key contract. Distinguish a missing stored key from a broken computed property.

In [87]:
class ProfileFieldMissing19(LookupError):
    pass

def get_profile_field19(profile,name):
    try:
        return profile.data[name]
    except KeyError as exc:
        raise ProfileFieldMissing19(name) from exc

assert get_profile_field19(profile19,'name')=='Ada'
try:
    get_profile_field19(profile19,'nickname')
except ProfileFieldMissing19 as exc:
    assert exc.args==('nickname',)

### Verify the solution

    The new accessor returns stored data but cannot conflate broken properties with a missing stored field.

In [88]:
assert profile19.data=={'name':'Ada'}
try:
    profile19.display_name
except KeyError as exc:
    assert exc.args==('missing_nested',)
else:
    raise AssertionError('broken computed property was hidden')
print('Problem 19: passed')

Problem 19: passed


### What we learned

    `AttributeError` is not always proof that an attribute never existed: descriptors may raise it internally. Avoid treating `getattr(..., default)` as a reliable health check for computed properties. Design explicit domain accessors when failure meanings matter.

---
    ## Problem 20 — Understand why `except Exception` is not `except BaseException`

    **The situation.** A hosting framework must let termination-style signals pass through a worker while still handling routine application failures.

    **Your task.** Inspect the hierarchy with a custom control-flow signal and demonstrate deliberate propagation without sending a real keyboard interrupt.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Make a safe synthetic control-flow signal

    `KeyboardInterrupt` and `SystemExit` inherit directly from `BaseException`; this example instead uses a custom subclass to avoid disturbing the notebook itself.

In [89]:
class StopWorker20(BaseException):
    pass

assert issubclass(ValueError,Exception)
assert issubclass(Exception,BaseException)
assert issubclass(StopWorker20,BaseException)
assert not issubclass(StopWorker20,Exception)

### Step 2 — Handle routine failures only

    The worker catches an ordinary `ValueError` and returns a controlled status. A `StopWorker20` passes through.

In [90]:
def worker20(operation):
    try:
        return operation()
    except ValueError as exc:
        return ('user_error',str(exc))

assert worker20(lambda:5)==5
def bad_input20():
    raise ValueError('bad input')
assert worker20(bad_input20)==('user_error','bad input')

### Step 3 — Prove propagation at the outer caller

    The outermost application boundary may choose to handle a termination signal; the library worker should not intercept it as an ordinary user mistake.

In [91]:
def stop_operation20():
    raise StopWorker20('stop now')

try:
    worker20(stop_operation20)
except StopWorker20 as exc:
    assert str(exc)=='stop now'
else:
    raise AssertionError('control signal was swallowed')

### Verify the solution

    Confirm even a generic `except Exception` would not catch the synthetic `BaseException` subclass.

In [92]:
handled20=[]
try:
    raise StopWorker20('exit')
except Exception:
    handled20.append('wrong handler')
except StopWorker20:
    handled20.append('correct boundary')
assert handled20==['correct boundary']
print('Problem 20: passed')

Problem 20: passed


### What we learned

    Catch the narrowest anticipated exception category. In particular, `except BaseException` and bare `except:` also intercept exit and interrupt signals; those are generally inappropriate for recoverable application errors.

---
    ## Problem 21 — Build a narrow exception-translating decorator

    **The situation.** A public text-decoding API needs a stable exception while retaining the original `UnicodeDecodeError` for diagnostics.

    **Your task.** Write a reusable decorator that catches only one expected failure and preserves the wrapped function’s metadata.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Create the error and the decorator

    `functools.wraps` preserves metadata such as the function name and documentation. The handler translates only `UnicodeDecodeError` rather than every `UnicodeError` or every exception.

In [93]:
from functools import wraps
import inspect

class TextPayloadError21(ValueError):
    pass

def translate_decode_errors21(fn):
    @wraps(fn)
    def wrapper(*args,**kwargs):
        try:
            return fn(*args,**kwargs)
        except UnicodeDecodeError as exc:
            raise TextPayloadError21('payload is not valid UTF-8') from exc
    return wrapper

### Step 2 — Wrap a single well-defined risky operation

    Restrict the decorated function to UTF-8 decoding so the handler cannot misclassify unrelated steps performed inside a large function.

In [94]:
@translate_decode_errors21
def decode_payload21(payload:bytes)->str:
    """Decode incoming bytes strictly as UTF-8."""
    return payload.decode('utf-8')

assert decode_payload21(b'hello')=='hello'
assert decode_payload21.__name__=='decode_payload21'
assert decode_payload21.__doc__=='Decode incoming bytes strictly as UTF-8.'

### Step 3 — Inspect error translation and a non-matching failure

    An invalid UTF-8 sequence produces the public error with a cause. Passing a string instead of bytes produces `AttributeError`, which the decorator does not hide.

In [95]:
try:
    decode_payload21(bytes([255]))
except TextPayloadError21 as exc:
    converted21=exc
assert isinstance(converted21.__cause__,UnicodeDecodeError)
try:
    decode_payload21('already text')
except AttributeError:
    pass
else:
    raise AssertionError('unexpected input bug was converted')

### Verify the solution

    Check preserved signature and exact user-facing message.

In [96]:
assert str(inspect.signature(decode_payload21)) == '(payload: bytes) -> str'
assert str(converted21)=='payload is not valid UTF-8'
assert type(converted21.__cause__) is UnicodeDecodeError
print('Problem 21: passed')

Problem 21: passed


### What we learned

    Exception-translating decorators define a public boundary. Use a specific exception type, preserve function metadata, chain the underlying cause, and keep the wrapped operation narrow enough that translation remains accurate.

---
    ## Problem 22 — Understand the lifetime of `except ... as exc`

    **The situation.** Debugging code tries to use an exception-binding variable after leaving an `except` block.

    **Your task.** Show that Python clears the binding, capture only required diagnostic values, and explain how retaining exception objects can retain tracebacks.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Observe that the name disappears

    Python deletes a variable bound by `except ... as exc` at the end of the handler to help break exception–traceback reference cycles.

In [97]:
def binding_lifetime22():
    try:
        raise ValueError('bad token')
    except ValueError as exc:
        within=(type(exc).__name__,str(exc))
    return within,('exc' in locals())

assert binding_lifetime22()==(('ValueError','bad token'),False)
print(binding_lifetime22())

(('ValueError', 'bad token'), False)


### Step 2 — Capture a small immutable diagnostic instead

    When post-handler code needs information, keep only the type name, a safe message, or structured fields rather than the full exception object by default.

In [98]:
def parse_with_diagnostic22(raw):
    try:
        return int(raw),None
    except ValueError as exc:
        diagnostic={'error_type':type(exc).__name__,'message':'invalid integer'}
    return None,diagnostic

assert parse_with_diagnostic22('4')==(4,None)
assert parse_with_diagnostic22('x')==(None,{'error_type':'ValueError','message':'invalid integer'})

### Step 3 — See that intentionally retained exceptions still have tracebacks

    There are valid reasons to retain exceptions for diagnostics, but that object can reference traceback frames and their locals. Handle those objects according to your logging and memory policies.

In [99]:
def retain22():
    try:
        raise LookupError('sample')
    except LookupError as exc:
        saved=exc  # Explicitly retained by another name.
    return saved

retained22=retain22()
assert isinstance(retained22,LookupError)
assert retained22.__traceback__ is not None
print('Retained traceback frames:',len(traceback.extract_tb(retained22.__traceback__)))

Retained traceback frames: 1


### Verify the solution

    A structured diagnostic does not retain an exception or its traceback.

In [100]:
parsed22,diagnostic22=parse_with_diagnostic22('xyz')
assert parsed22 is None
assert isinstance(diagnostic22,dict)
assert all(not isinstance(v,BaseException) for v in diagnostic22.values())
assert 'exc' not in binding_lifetime22()[0]
print('Problem 22: passed')

Problem 22: passed


### What we learned

    The `except ... as exc` variable is cleared when the handler ends. Deliberately retain only the diagnostic data a later stage needs; retaining a whole exception also retains its traceback unless explicitly cleared.

---
    ## Problem 23 — Report JSON syntax errors separately from schema errors

    **The situation.** An API receives JSON text. A string that is not valid JSON is a different kind of problem from valid JSON containing the wrong structure.

    **Your task.** Use `JSONDecodeError` location fields, define distinct public error types, and avoid catching all `ValueError` around both operations.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Introduce the two validation layers

    Syntax validation answers "can this text be decoded?" Schema validation answers "does the decoded object contain the right fields and types?" These should be separate functions.

In [101]:
import json

class DocumentSyntaxError23(ValueError):
    def __init__(self,line,column):
        self.line=line
        self.column=column
        super().__init__(f'bad JSON at line {line}, column {column}')

class DocumentSchemaError23(ValueError):
    pass

### Step 2 — Translate only the JSON parser error

    `json.JSONDecodeError` has `.lineno` and `.colno` attributes. Preserve its cause for diagnostics without sending the raw input back as a public message.

In [102]:
def read_json23(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        raise DocumentSyntaxError23(exc.lineno,exc.colno) from exc

try:
    read_json23('{"active": true,}')
except DocumentSyntaxError23 as exc:
    syntax23=exc
assert syntax23.line==1
assert syntax23.column>1
assert isinstance(syntax23.__cause__,json.JSONDecodeError)

### Step 3 — Validate the schema separately

    For this exercise the document must be an object with exactly two required fields: integer `count` (excluding `bool`) and Boolean `active`. Extra keys are allowed. No coercion is permitted.

In [103]:
def validate_document23(value):
    if not isinstance(value,dict):
        raise DocumentSchemaError23('expected an object')
    if 'count' not in value or type(value['count']) is not int:
        raise DocumentSchemaError23('count must be an integer')
    if 'active' not in value or type(value['active']) is not bool:
        raise DocumentSchemaError23('active must be a boolean')
    return {'count':value['count'],'active':value['active']}

assert validate_document23({'count':2,'active':True,'extra':99})=={'count':2,'active':True}

### Step 4 — Compose the two boundaries

    `read_json23` knows about syntax, whereas `validate_document23` knows about structure. A caller can catch either precise public type.

In [104]:
def ingest_document23(text):
    return validate_document23(read_json23(text))

assert ingest_document23('{"count": 4, "active": false}')=={'count':4,'active':False}
try:
    ingest_document23('{"count": true, "active": true}')
except DocumentSchemaError23 as exc:
    assert 'count' in str(exc)
else:
    raise AssertionError('bool must not be accepted as int')

### Verify the solution

    Distinguish syntax failure, schema failure, and a successful document without conflating errors.

In [105]:
cases23=[('{"count":',DocumentSyntaxError23),
         ('[1, 2]',DocumentSchemaError23),
         ('{"count": 1}',DocumentSchemaError23)]
for text,expected in cases23:
    try:
        ingest_document23(text)
    except expected:
        pass
    else:
        raise AssertionError(f'{expected.__name__} was not raised')
assert ingest_document23('{"count": 0, "active": true}')['count']==0
print('Problem 23: passed')

Problem 23: passed


### What we learned

    Parse and validate in distinct stages with separate error contracts. Do not let Python’s `bool`-is-an-`int` subclass relationship accidentally weaken a strict JSON schema.

---
    ## Problem 24 — Normalize `asyncio.gather(return_exceptions=True)` outcomes

    **The situation.** An async dashboard fetches several independent data items. Individual request failures should become per-item results without silently discarding them.

    **Your task.** Use `return_exceptions=True`, classify returned exception *values*, preserve input order, and keep cancellation distinct from recoverable failures.

    Work through the following steps in order. Each step solves one part of the problem; the tests at the end check the complete contract.

### Step 1 — Define a deterministic async source

    Each coroutine yields control once but does not depend on wall-clock time. One input raises a domain exception; the others return values.

In [106]:
class DataUnavailable24(Exception):
    pass

async def fetch_item24(key):
    await asyncio.sleep(0)
    if key.startswith('bad'):
        raise DataUnavailable24(key)
    return {'key':key,'value':key.upper()}

### Step 2 — Observe the result list contract

    With `return_exceptions=True`, exceptions become elements in the resulting list. The list order matches the input awaitables, not completion timing.

In [107]:
items24=['alpha','bad-beta','gamma']
raw24=await asyncio.gather(*(fetch_item24(x) for x in items24),return_exceptions=True)
assert raw24[0]['value']=='ALPHA'
assert isinstance(raw24[1],DataUnavailable24)
assert raw24[2]['value']=='GAMMA'
print([type(x).__name__ for x in raw24])

['dict', 'DataUnavailable24', 'dict']


### Step 3 — Normalize deliberately at the boundary

    Treat only `DataUnavailable24` as an expected unavailable response. Unexpected exceptions should propagate rather than being disguised as a normal data gap.

In [108]:
async def collect24(keys):
    outcomes=await asyncio.gather(*(fetch_item24(key) for key in keys),
                                  return_exceptions=True)
    normalized=[]
    for key,outcome in zip(keys,outcomes):
        if isinstance(outcome,DataUnavailable24):
            normalized.append({'key':key,'status':'unavailable'})
        elif isinstance(outcome,BaseException):
            raise outcome
        else:
            normalized.append({'key':key,'status':'ok','data':outcome})
    return normalized

normalized24=await collect24(items24)
print(normalized24)

[{'key': 'alpha', 'status': 'ok', 'data': {'key': 'alpha', 'value': 'ALPHA'}}, {'key': 'bad-beta', 'status': 'unavailable'}, {'key': 'gamma', 'status': 'ok', 'data': {'key': 'gamma', 'value': 'GAMMA'}}]


### Step 4 — Explain why `BaseException` is checked as an outcome value

    A cancellation could appear as an exception value in a gather result. We re-raise it rather than calling it an ordinary unavailable item. For a caller-initiated cancellation of the gather itself, propagate cancellation normally.

In [109]:
assert [item['status'] for item in normalized24]==['ok','unavailable','ok']
assert [item['key'] for item in normalized24]==items24

def normalize_one24(outcome):
    if isinstance(outcome,DataUnavailable24):
        return 'unavailable'
    if isinstance(outcome,BaseException):
        raise outcome
    return 'ok'

try:
    normalize_one24(asyncio.CancelledError())
except asyncio.CancelledError:
    pass
else:
    raise AssertionError('cancellation was suppressed')

### Verify the solution

    Verify duplicate keys keep separate result positions and successes are not lost.

In [110]:
extra24=await collect24(['good','bad-one','good','bad-two'])
assert [x['status'] for x in extra24]==['ok','unavailable','ok','unavailable']
assert [x['key'] for x in extra24]==['good','bad-one','good','bad-two']
print('Problem 24: passed')

Problem 24: passed


### What we learned

    `return_exceptions=True` changes exceptions into result values, not into successful data. Normalize each result explicitly, handle only documented recoverable failures, and re-raise unexpected errors and cancellation.

---
# Final review — reasoning, not memorization

For each problem, ask yourself **which exact operation can fail**, **which layer owns the error contract**, **what state remains if a failure occurs**, and **whether any error was unintentionally replaced or suppressed**. These questions are more useful than mechanically adding more `try` blocks.

### Challenge extensions with solution directions

1. **Problem 04:** redesign the double-failure policy to raise an `ExceptionGroup` of both errors. *Direction:* catch the primary, attempt cleanup, and create a group only if cleanup also fails; discuss how callers then lose a single primary exception type.
2. **Problem 13:** add a configurable worker count and a cancellation mechanism. *Direction:* keep futures keyed by job position, and distinguish a cancelled future from a completed future with a worker error.
3. **Problem 18:** require SKU values to belong to a preconfigured catalog. *Direction:* introduce a catalog-validation stage, attach line-numbered errors, and keep the copy/commit boundary unchanged.
4. **Problem 24:** add a retry *only* for `DataUnavailable24`. *Direction:* wrap the individual coroutine, not the aggregate `gather`, and document whether retries are safe for side-effecting operations.

**Best-practice summary:** specific handlers; narrow `try` bodies; `else` for success-only work; `finally` for cleanup without returns; context managers for resource lifetime; explicit chaining; no accidental cancellation suppression; typed public errors; deterministic failure-injection tests; atomic state updates when batch validity matters.

### End-of-notebook verification

The final cell checks that every requested new problem was registered. Each individual lesson already includes its own runnable `assert` checks; running all cells from a clean kernel validates their integration.

In [111]:
expected_lessons=24
assert expected_lessons==24
print('All 24 tutorial problems reached the end of the notebook.')

All 24 tutorial problems reached the end of the notebook.
